# Fusion des datasets annuels

À lancer après avoir généré les années voulues avec `dataset_annee.ipynb`.
Fusionne `data/processed/annees/trafic_*.csv` en un seul fichier historique,
puis (optionnel) y ajoute les données récentes `trafic_3_arcs.csv`.

In [2]:
from pathlib import Path

import pandas as pd

ANNEES_DIR = Path("data/processed/annees")
RECENT = Path("data/processed/trafic_3_arcs.csv")
OUT_HISTO = Path("data/processed/trafic_3_arcs_historique.csv")
OUT_COMPLET = Path("data/processed/trafic_3_arcs_complet.csv")

## 1. Chargement et concaténation des années

In [3]:
fichiers = sorted(ANNEES_DIR.glob("trafic_*.csv"))
print("Années trouvées :", [f.stem.split("_")[-1] for f in fichiers])

histo = pd.concat([pd.read_csv(f) for f in fichiers], ignore_index=True)
histo["datetime_utc"] = pd.to_datetime(histo["datetime_utc"], utc=True)

# Les fichiers se chevauchent de quelques heures au 1er janvier :
# on garde en priorité la ligne qui contient une vraie mesure
n_avant = len(histo)
histo = (
    histo.sort_values(["axe", "datetime_utc", "ligne_absente"])
    .drop_duplicates(subset=["axe", "datetime_utc"], keep="first")
    .reset_index(drop=True)
)
print(f"{n_avant - len(histo)} doublons entre années supprimés")

Années trouvées : ['2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']
0 doublons entre années supprimés


## 2. Grille horaire complète sur toute la période (comble aussi les années non générées)

In [4]:
grille_utc = pd.date_range(histo["datetime_utc"].min(), histo["datetime_utc"].max(), freq="h")

complet = []
for axe, g in histo.groupby("axe"):
    arc_id = g["arc_id"].mode().iat[0]
    g = g.set_index("datetime_utc").reindex(grille_utc)
    g["axe"] = axe
    g["arc_id"] = arc_id
    g["ligne_absente"] = g["ligne_absente"].fillna(True).astype(bool)
    complet.append(g)

histo = pd.concat(complet).rename_axis("datetime_utc").reset_index()
histo["datetime"] = histo["datetime_utc"].dt.tz_convert("Europe/Paris")
histo = histo[["axe", "arc_id", "datetime", "datetime_utc", "debit", "occupation",
               "etat_trafic", "etat_arc_code", "ligne_absente"]]
print(f"{len(histo)} lignes, du {histo['datetime'].min()} au {histo['datetime'].max()}")

315576 lignes, du 2014-01-01 01:00:00+01:00 au 2026-01-01 00:00:00+01:00


## 3. Bilan par axe et par année (% de débit manquant, débit moyen)

In [5]:
annee = histo["datetime"].dt.year
display(histo.groupby(["axe", annee])["debit"].apply(lambda s: round(s.isna().mean() * 100, 1)).unstack("axe"))
display(histo.groupby(["axe", annee])["debit"].mean().round(0).unstack("axe"))

axe,Champs-Elysees,Convention,Saints-Peres
datetime,,,
2014,1.0,2.0,0.6
2015,1.8,1.7,1.2
2016,1.5,1.4,1.1
2017,11.3,3.3,0.7
2018,1.9,1.1,0.7
2019,27.5,1.0,13.9
2020,0.7,2.6,0.7
2021,0.8,3.4,0.7
2022,14.3,19.1,4.0


axe,Champs-Elysees,Convention,Saints-Peres
datetime,,,
2014,1469.0,346.0,554.0
2015,1424.0,347.0,567.0
2016,1327.0,504.0,531.0
2017,1291.0,544.0,494.0
2018,1127.0,627.0,381.0
2019,808.0,622.0,481.0
2020,686.0,512.0,364.0
2021,769.0,527.0,392.0
2022,858.0,518.0,408.0


In [6]:
# datetime (heure de Paris) n'est pas sauvegardée : elle se recalcule depuis datetime_utc
histo.drop(columns="datetime").to_csv(OUT_HISTO, index=False)
print(f"Sauvegardé : {OUT_HISTO}  ({len(histo)} lignes)")

Sauvegardé : data\processed\trafic_3_arcs_historique.csv  (315576 lignes)


## 4. Vérification : l'historique est-il cohérent avec les données récentes ?

`old_data/2025` et `trafic_3_arcs.csv` se recouvrent (fin 2025). Si les fuseaux horaires
sont bien gérés, le débit doit être quasi identique **sans décalage** (corrélation max à 0 h).

In [7]:
recent = pd.read_csv(RECENT)
recent["datetime_utc"] = pd.to_datetime(recent["datetime_utc"], utc=True)

for axe in sorted(recent["axe"].unique()):
    a = histo[histo["axe"] == axe].set_index("datetime_utc")["debit"]
    b = recent[recent["axe"] == axe].set_index("datetime_utc")["debit"]
    corr = {f"{d:+d}h": round(a.shift(d, freq="h").corr(b), 3) for d in (-2, -1, 0, 1, 2)}
    n = a.dropna().index.intersection(b.dropna().index).size
    print(f"{axe:15s} heures communes={n:5d}  corrélation selon décalage : {corr}")

Champs-Elysees  heures communes= 1141  corrélation selon décalage : {'-2h': np.float64(0.695), '-1h': np.float64(0.873), '+0h': np.float64(1.0), '+1h': np.float64(0.873), '+2h': np.float64(0.695)}
Convention      heures communes= 1828  corrélation selon décalage : {'-2h': np.float64(0.776), '-1h': np.float64(0.913), '+0h': np.float64(1.0), '+1h': np.float64(0.913), '+2h': np.float64(0.775)}
Saints-Peres    heures communes= 1681  corrélation selon décalage : {'-2h': np.float64(0.793), '-1h': np.float64(0.925), '+0h': np.float64(1.0), '+1h': np.float64(0.926), '+2h': np.float64(0.794)}


## 5. (Optionnel) Dataset complet : historique + données récentes

Sur la période commune, on garde les **données récentes** (source officielle du challenge).
Attention : l'état de l'arc est `etat_arc` (texte) dans le récent et `etat_arc_code` (code) dans l'historique.

In [8]:
recent["datetime"] = recent["datetime_utc"].dt.tz_convert("Europe/Paris")
debut_recent = recent["datetime_utc"].min()

full = pd.concat(
    [histo[histo["datetime_utc"] < debut_recent].assign(source="old_data"),
     recent.assign(source="recent")],
    ignore_index=True,
).sort_values(["axe", "datetime_utc"]).reset_index(drop=True)

full.drop(columns="datetime").to_csv(OUT_COMPLET, index=False)  # idem : datetime se recalcule depuis datetime_utc
print(f"Sauvegardé : {OUT_COMPLET}  ({len(full)} lignes)")
full.groupby(["axe", "source"])["datetime"].agg(["min", "max", "size"])

Sauvegardé : data\processed\trafic_3_arcs_complet.csv  (337464 lignes)


min                       max  \
axe            source                                                         
Champs-Elysees old_data 2014-01-01 01:00:00+01:00 2025-10-01 04:00:00+02:00   
               recent   2025-10-01 05:00:00+02:00 2026-11-01 00:00:00+01:00   
Convention     old_data 2014-01-01 01:00:00+01:00 2025-10-01 04:00:00+02:00   
               recent   2025-10-01 05:00:00+02:00 2026-11-01 00:00:00+01:00   
Saints-Peres   old_data 2014-01-01 01:00:00+01:00 2025-10-01 04:00:00+02:00   
               recent   2025-10-01 05:00:00+02:00 2026-11-01 00:00:00+01:00   

                           size  
axe            source            
Champs-Elysees old_data  102987  
               recent      9501  
Convention     old_data  102987  
               recent      9501  
Saints-Peres   old_data  102987  
               recent      9501

In [9]:
full

,axe,arc_id,datetime,datetime_utc,debit,occupation,etat_trafic,etat_arc_code,ligne_absente,source,etat_arc
0,Champs-Elysees,4264,2014-01-01 01:00:00+01:00,2014-01-01 00:00:00+00:00,4.0,0.00000,Fluide,3.0,False,old_data,NaN
1,Champs-Elysees,4264,2014-01-01 02:00:00+01:00,2014-01-01 01:00:00+00:00,21.0,0.10445,Fluide,3.0,False,old_data,NaN
2,Champs-Elysees,4264,2014-01-01 03:00:00+01:00,2014-01-01 02:00:00+00:00,417.0,6.12667,Fluide,3.0,False,old_data,NaN
3,Champs-Elysees,4264,2014-01-01 04:00:00+01:00,2014-01-01 03:00:00+00:00,1089.0,14.39890,Fluide,3.0,False,old_data,NaN
4,Champs-Elysees,4264,2014-01-01 05:00:00+01:00,2014-01-01 04:00:00+00:00,943.0,10.35220,Fluide,3.0,False,old_data,NaN
...,...,...,...,...,...,...,...,...,...,...,...
337459,Saints-Peres,191,2026-10-31 20:00:00+01:00,2026-10-31 19:00:00+00:00,463.0,6.04056,Fluide,NaN,False,recent,Ouvert
337460,Saints-Peres,191,2026-10-31 21:00:00+01:00,2026-10-31 20:00:00+00:00,492.0,6.05778,Fluide,NaN,False,recent,Ouvert
337461,Saints-Peres,191,2026-10-31 22:00:00+01:00,2026-10-31 21:00:00+00:00,439.0,5.36889,Fluide,NaN,False,recent,Ouvert
337462,Saints-Peres,191,2026-10-31 23:00:00+01:00,2026-10-31 22:00:00+00:00,412.0,5.02500,Fluide,NaN,False,recent,Ouvert
